# VLM-валидация детекции курения

Пайплайн:
1. Базовый инференс: cigarette_model + smoke_model с буст-коэффициентом при пересечении рамок
2. Апскейлинг детектированных патчей через Real-ESRGAN / LANCZOS4 с адаптивным выбором метода
3. VLM-эксперименты: CLIP RN50, SigLIP ViT-B/16, CLIP ViT-L/14 — подбор порогов и промптов
4. Дистилляция: обучение лёгкого MLP-классификатора на CLIP-эмбеддингах
5. Финальная оценка: Precision / Recall / F1 на micro и small датасетах сигарет и дыма
6. Сравнительная таблица всех конфигураций

## 0. Установка зависимостей

In [ ]:
!pip install -q open-clip-torch ultralytics huggingface_hub torch torchvision
!pip install -q scikit-learn matplotlib pandas tqdm Pillow opencv-python-headless
# Real-ESRGAN для качественного апскейлинга
!pip install -q basicsr facexlib gfpgan
!pip install -q realesrgan

## 1. Импорты

In [ ]:
import copy
import json
import logging
import os
import random
import time
import warnings

import cv2
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import open_clip
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from huggingface_hub import hf_hub_download
from pathlib import Path
from PIL import Image
from scipy.optimize import linear_sum_assignment
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from ultralytics import YOLO

logging.getLogger('ultralytics').setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

## 2. Конфигурация

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# --- пути к моделям ---
CIGARETTE_MODEL_PATH = 'runs/detect/runs/detect/smoking_yolo26l/weights/best.pt'
SMOKE_MODEL_PATH     = 'runs/detect/smoke_boxes_micro_s/smoking_yolo26s_micro/weights/best.pt'

# --- пути к датасетам (из comparison-ноутбуков) ---
SMALL_DATASET_DIR = Path('yolo_dataset_small')
MICRO_DATASET_DIR = Path('yolo_dataset')
SMOKE_SMALL_DIR   = Path('smoke_yolo_dataset')
SMOKE_MICRO_DIR   = Path('smoke_yolo_dataset_micro')

# --- пороги базового пайплайна ---
CIG_CONF_THRESH      = 0.20   # низкий порог для повышения полноты
SMOKE_CONF_THRESH    = 0.20
SMOKE_IOU_BOOST_MIN  = 0.05   # при IoU > этого порога включается буст
BOOST_ALPHA          = 0.35   # conf_final = conf_cig * (1 + alpha * max_iou)
FINAL_CONF_THRESH    = 0.25   # после буста — итоговый порог

# --- апскейлинг ---
UPSCALE_MIN_SIDE     = 112    # если min(h,w) < этого — апскейлить
UPSCALE_TARGET_SIDE  = 224    # целевой размер меньшей стороны
EXPAND_FACTOR        = 1.6    # коэффициент расширения рамки перед VLM

# --- CLIP / VLM ---
CLIP_BATCH_SIZE      = 64
EMBED_CACHE_DIR      = Path('vlm_embed_cache')
EMBED_CACHE_DIR.mkdir(exist_ok=True)

# --- дистиллированный классификатор ---
CLASSIFIER_EPOCHS    = 30
CLASSIFIER_LR        = 3e-4
CLASSIFIER_HIDDEN    = 256

OUTPUT_DIR = Path('vlm_validation_results')
OUTPUT_DIR.mkdir(exist_ok=True)

## 3. DetectionTracker (скопирован из experiment_inference_final)

In [ ]:
def compute_iou(box_a, box_b):
    xa = max(box_a[0], box_b[0])
    ya = max(box_a[1], box_b[1])
    xb = min(box_a[2], box_b[2])
    yb = min(box_a[3], box_b[3])
    inter = max(0, xb - xa) * max(0, yb - ya)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def compute_distance(box_a, box_b):
    cx_a = (box_a[0] + box_a[2]) / 2
    cy_a = (box_a[1] + box_a[3]) / 2
    cx_b = (box_b[0] + box_b[2]) / 2
    cy_b = (box_b[1] + box_b[3]) / 2
    return ((cx_a - cx_b) ** 2 + (cy_a - cy_b) ** 2) ** 0.5


class DetectionTracker:
    def __init__(self, max_age=5, min_hits=2, iou_threshold=0.3, distance_threshold=200):
        self.max_age = max_age
        self.min_hits = min_hits
        self.iou_threshold = iou_threshold
        self.distance_threshold = distance_threshold
        self.tracks = []
        self.next_id = 0

    def update(self, detections):
        if len(self.tracks) == 0:
            for det in detections:
                self.tracks.append({
                    'id': self.next_id,
                    'box': det['box'],
                    'conf': det['conf'],
                    'cls': det['cls'],
                    'age': 0,
                    'hits': 1,
                    'conf_history': [det['conf']]
                })
                self.next_id += 1
            return self.tracks

        if len(detections) == 0:
            for track in self.tracks:
                track['age'] += 1
            self.tracks = [t for t in self.tracks if t['age'] < self.max_age]
            return self.tracks

        iou_matrix = np.zeros((len(self.tracks), len(detections)))
        for i, track in enumerate(self.tracks):
            for j, det in enumerate(detections):
                iou = compute_iou(track['box'], det['box'])
                dist = compute_distance(track['box'], det['box'])
                if dist < self.distance_threshold:
                    iou_matrix[i, j] = iou

        row_ind, col_ind = linear_sum_assignment(-iou_matrix)

        matched_tracks = set()
        matched_dets = set()

        for i, j in zip(row_ind, col_ind):
            if iou_matrix[i, j] > self.iou_threshold:
                self.tracks[i]['box'] = detections[j]['box']
                self.tracks[i]['conf'] = detections[j]['conf']
                self.tracks[i]['cls'] = detections[j]['cls']
                self.tracks[i]['age'] = 0
                self.tracks[i]['hits'] += 1
                self.tracks[i]['conf_history'].append(detections[j]['conf'])
                if len(self.tracks[i]['conf_history']) > 10:
                    self.tracks[i]['conf_history'].pop(0)
                matched_tracks.add(i)
                matched_dets.add(j)

        for i, track in enumerate(self.tracks):
            if i not in matched_tracks:
                track['age'] += 1

        for j, det in enumerate(detections):
            if j not in matched_dets:
                self.tracks.append({
                    'id': self.next_id,
                    'box': det['box'],
                    'conf': det['conf'],
                    'cls': det['cls'],
                    'age': 0,
                    'hits': 1,
                    'conf_history': [det['conf']]
                })
                self.next_id += 1

        self.tracks = [t for t in self.tracks if t['age'] < self.max_age]
        return [t for t in self.tracks if t['hits'] >= self.min_hits]

## 4. Загрузка YOLO-моделей

In [ ]:
cigarette_model = YOLO(CIGARETTE_MODEL_PATH)
smoke_model     = YOLO(SMOKE_MODEL_PATH)

print('cigarette_model classes:', cigarette_model.names)
print('smoke_model classes:    ', smoke_model.names)

## 5. Апскейлинг патчей

Стратегия: если меньшая сторона кропа < UPSCALE_MIN_SIDE — применяем апскейлинг.
- Очень маленькие объекты (min_side < 32): Real-ESRGAN x4 (если доступен), иначе LANCZOS4
- Умеренно маленькие (32..UPSCALE_MIN_SIDE): LANCZOS4
- Остальные: просто resize до UPSCALE_TARGET_SIDE по меньшей стороне

In [ ]:
try:
    from basicsr.archs.rrdbnet_arch import RRDBNet
    from realesrgan import RealESRGANer

    _esrgan_model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                            num_block=23, num_grow_ch=32, scale=4)
    _esrgan_weights = hf_hub_download(
        repo_id='ai-forever/Real-ESRGAN',
        filename='RealESRGAN_x4plus.pth'
    )
    _upsampler = RealESRGANer(
        scale=4,
        model_path=_esrgan_weights,
        model=_esrgan_model,
        tile=256,
        tile_pad=10,
        pre_pad=0,
        half=True if DEVICE == 'cuda' else False,
        device=DEVICE
    )
    ESRGAN_AVAILABLE = True
    print('Real-ESRGAN загружен')
except Exception as _e:
    ESRGAN_AVAILABLE = False
    print(f'Real-ESRGAN недоступен ({_e}), будет использован LANCZOS4')

In [ ]:
def expand_box(box, frame_h, frame_w, expand_factor=EXPAND_FACTOR):
    x1, y1, x2, y2 = map(int, box)
    bw = x2 - x1
    bh = y2 - y1
    pad_x = int(bw * (expand_factor - 1) / 2)
    pad_y = int(bh * (expand_factor - 1) / 2)
    x1e = max(0, x1 - pad_x)
    y1e = max(0, y1 - pad_y)
    x2e = min(frame_w, x2 + pad_x)
    y2e = min(frame_h, y2 + pad_y)
    return x1e, y1e, x2e, y2e


def adaptive_upscale(crop_bgr, min_side=UPSCALE_MIN_SIDE, target_side=UPSCALE_TARGET_SIDE):
    h, w = crop_bgr.shape[:2]
    short = min(h, w)

    if short >= target_side:
        return crop_bgr

    if short < 32 and ESRGAN_AVAILABLE:
        try:
            out, _ = _upsampler.enhance(crop_bgr, outscale=4)
            h2, w2 = out.shape[:2]
            short2 = min(h2, w2)
            if short2 > target_side:
                scale_down = target_side / short2
                out = cv2.resize(out, (int(w2 * scale_down), int(h2 * scale_down)),
                                 interpolation=cv2.INTER_AREA)
            return out
        except Exception:
            pass

    scale = target_side / short
    new_w = int(w * scale)
    new_h = int(h * scale)
    return cv2.resize(crop_bgr, (new_w, new_h), interpolation=cv2.INTER_LANCZOS4)


def crop_and_upscale(frame_bgr, box, expand_factor=EXPAND_FACTOR):
    h, w = frame_bgr.shape[:2]
    x1, y1, x2, y2 = expand_box(box, h, w, expand_factor)
    crop = frame_bgr[y1:y2, x1:x2]
    if crop.size == 0:
        return None
    return adaptive_upscale(crop)

## 6. Базовый YOLO-пайплайн: инференс + буст от дыма

In [ ]:
def yolo_detect_image(img_path, cig_conf=CIG_CONF_THRESH, smoke_conf=SMOKE_CONF_THRESH,
                      boost_alpha=BOOST_ALPHA, smoke_iou_min=SMOKE_IOU_BOOST_MIN,
                      final_conf=FINAL_CONF_THRESH):
    img = cv2.imread(str(img_path))
    if img is None:
        return [], img

    cig_res   = cigarette_model.predict(img, conf=cig_conf,   verbose=False, device=DEVICE)
    smoke_res = smoke_model.predict(img,     conf=smoke_conf,  verbose=False, device=DEVICE)

    cig_dets = []
    if len(cig_res[0].boxes) > 0:
        for box, conf, cls in zip(cig_res[0].boxes.xyxy.cpu().numpy(),
                                   cig_res[0].boxes.conf.cpu().numpy(),
                                   cig_res[0].boxes.cls.cpu().numpy().astype(int)):
            cig_dets.append({'box': box.tolist(), 'conf': float(conf), 'cls': int(cls)})

    smoke_dets = []
    if len(smoke_res[0].boxes) > 0:
        for box in smoke_res[0].boxes.xyxy.cpu().numpy():
            smoke_dets.append(box.tolist())

    merged = []
    for det in cig_dets:
        max_smoke_iou = 0.0
        for s_box in smoke_dets:
            iou = compute_iou(det['box'], s_box)
            if iou > max_smoke_iou:
                max_smoke_iou = iou
        boost = 0.0
        if max_smoke_iou > smoke_iou_min:
            boost = boost_alpha * max_smoke_iou
        final = min(det['conf'] * (1.0 + boost), 1.0)
        if final >= final_conf:
            merged.append({
                'box': det['box'],
                'conf': final,
                'cls': det['cls'],
                'has_smoke': max_smoke_iou > smoke_iou_min
            })

    return merged, img

## 7. Загрузка VLM-моделей

Протестируем три варианта:
- `CLIP RN50` (openai) — лёгкий, 38M параметров
- `CLIP ViT-L/14` (openai) — тяжелее, но лучшее качество embeddings
- `SigLIP ViT-B/16` (google) — современный, хорошо работает с маленькими патчами

In [ ]:
VLM_CONFIGS = [
    {'name': 'CLIP_RN50',      'model': 'RN50',          'pretrained': 'openai'},
    {'name': 'CLIP_ViTL14',    'model': 'ViT-L-14',      'pretrained': 'openai'},
    {'name': 'SigLIP_ViTB16',  'model': 'ViT-B-16-SigLIP', 'pretrained': 'webli'},
]

loaded_vlms = {}

for cfg in VLM_CONFIGS:
    try:
        model, _, preprocess = open_clip.create_model_and_transforms(
            cfg['model'], pretrained=cfg['pretrained']
        )
        model = model.to(DEVICE).eval()
        tokenizer = open_clip.get_tokenizer(cfg['model'])
        loaded_vlms[cfg['name']] = {
            'model': model,
            'preprocess': preprocess,
            'tokenizer': tokenizer,
        }
        print(f'Загружен {cfg["name"]}')
    except Exception as e:
        print(f'Не удалось загрузить {cfg["name"]}: {e}')

## 8. Текстовые промпты для VLM-классификации

In [ ]:
# positive: курит; negative: не курит
PROMPT_SETS = {
    'basic': {
        'pos': ['a person smoking a cigarette', 'cigarette', 'smoking'],
        'neg': ['a person not smoking', 'phone', 'bottle', 'pen', 'hand', 'nothing'],
    },
    'detailed': {
        'pos': [
            'a person holding a cigarette to their mouth',
            'cigarette smoke coming from a person',
            'a lit cigarette between fingers',
            'vaping e-cigarette',
        ],
        'neg': [
            'a person holding a phone',
            'a person holding a bottle',
            'a person holding a pen or pencil',
            'empty hand raised near face',
            'person eating food',
        ],
    },
    'binary': {
        'pos': ['smoking'],
        'neg': ['not smoking'],
    },
}


def encode_text_prompts(vlm_entry, prompt_set_name='basic'):
    pset = PROMPT_SETS[prompt_set_name]
    model     = vlm_entry['model']
    tokenizer = vlm_entry['tokenizer']
    all_texts = pset['pos'] + pset['neg']
    tokens = tokenizer(all_texts).to(DEVICE)
    with torch.no_grad():
        feats = model.encode_text(tokens)
        feats = F.normalize(feats, dim=-1)
    n_pos = len(pset['pos'])
    pos_feat = feats[:n_pos].mean(dim=0, keepdim=True)
    neg_feat = feats[n_pos:].mean(dim=0, keepdim=True)
    pos_feat = F.normalize(pos_feat, dim=-1)
    neg_feat = F.normalize(neg_feat, dim=-1)
    return pos_feat, neg_feat

## 9. VLM-верификация одного патча

In [ ]:
def vlm_score_patch(crop_bgr, vlm_entry, pos_feat, neg_feat):
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    pil_img  = Image.fromarray(crop_rgb)
    preprocess = vlm_entry['preprocess']
    model      = vlm_entry['model']
    inp = preprocess(pil_img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        img_feat = model.encode_image(inp)
        img_feat = F.normalize(img_feat, dim=-1)
    sim_pos = (img_feat @ pos_feat.T).item()
    sim_neg = (img_feat @ neg_feat.T).item()
    # вероятность через softmax
    logits = torch.tensor([sim_pos, sim_neg]) * 100.0
    prob_smoking = torch.softmax(logits, dim=0)[0].item()
    return prob_smoking


def vlm_verify_detections(frame_bgr, detections, vlm_name, prompt_set_name,
                           threshold=0.5, expand_factor=EXPAND_FACTOR):
    if vlm_name not in loaded_vlms:
        return [True] * len(detections)
    vlm_entry = loaded_vlms[vlm_name]
    pos_feat, neg_feat = encode_text_prompts(vlm_entry, prompt_set_name)
    h, w = frame_bgr.shape[:2]
    results = []
    for det in detections:
        crop = crop_and_upscale(frame_bgr, det['box'], expand_factor)
        if crop is None:
            results.append(False)
            continue
        score = vlm_score_patch(crop, vlm_entry, pos_feat, neg_feat)
        results.append(score >= threshold)
    return results

## 10. Сбор датасета эмбеддингов для дистилляции

Из тестовой выборки датасетов сигарет извлекаем патчи с GT-разметкой
и создаём бинарный датасет: smoking=1, not_smoking=0.
Для not_smoking берём случайные кропы изображений без аннотаций или фоновые патчи.

In [ ]:
def load_gt_boxes_yolo(label_path, img_w, img_h):
    boxes = []
    if not Path(label_path).exists():
        return boxes
    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls = int(float(parts[0]))
            xc, yc, bw, bh = map(float, parts[1:5])
            x1 = (xc - bw / 2) * img_w
            y1 = (yc - bh / 2) * img_h
            x2 = (xc + bw / 2) * img_w
            y2 = (yc + bh / 2) * img_h
            boxes.append({'box': [x1, y1, x2, y2], 'cls': cls})
    return boxes


def collect_embed_dataset(dataset_dir, vlm_name, prompt_set_name='basic',
                           split='test', max_pos=800, max_neg=800,
                           expand_factor=EXPAND_FACTOR, smoking_cls_ids=None):
    if smoking_cls_ids is None:
        smoking_cls_ids = {0}

    cache_key = f'{Path(dataset_dir).name}_{vlm_name}_{prompt_set_name}_{split}'
    cache_path = EMBED_CACHE_DIR / f'{cache_key}.npz'
    if cache_path.exists():
        print(f'Загружаю эмбеддинги из кеша: {cache_path}')
        data = np.load(cache_path)
        return data['X'], data['y']

    vlm_entry  = loaded_vlms[vlm_name]
    model      = vlm_entry['model']
    preprocess = vlm_entry['preprocess']

    images_dir = Path(dataset_dir) / split / 'images'
    labels_dir = Path(dataset_dir) / split / 'labels'
    if not images_dir.exists():
        print(f'Сплит {split} не найден в {dataset_dir}')
        return np.array([]), np.array([])

    img_paths = sorted(images_dir.glob('*.jpg')) + sorted(images_dir.glob('*.png'))
    random.shuffle(img_paths)

    X_pos, X_neg = [], []

    for img_path in tqdm(img_paths, desc=f'Embedding {vlm_name}/{split}'):
        if len(X_pos) >= max_pos and len(X_neg) >= max_neg:
            break

        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            continue
        h, w = img_bgr.shape[:2]

        label_path = labels_dir / (img_path.stem + '.txt')
        gt_boxes = load_gt_boxes_yolo(label_path, w, h)

        for gt in gt_boxes:
            if len(X_pos) >= max_pos:
                break
            if gt['cls'] not in smoking_cls_ids:
                continue
            crop = crop_and_upscale(img_bgr, gt['box'], expand_factor)
            if crop is None:
                continue
            crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            pil_img  = Image.fromarray(crop_rgb)
            inp = preprocess(pil_img).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                feat = model.encode_image(inp)
                feat = F.normalize(feat, dim=-1)
            X_pos.append(feat.cpu().numpy()[0])

        if len(gt_boxes) == 0 and len(X_neg) < max_neg:
            # фоновое изображение — берём случайный патч
            px = random.randint(0, max(0, w - 64))
            py = random.randint(0, max(0, h - 64))
            box = [px, py, min(w, px + 128), min(h, py + 128)]
            crop = crop_and_upscale(img_bgr, box, expand_factor=1.0)
            if crop is None:
                continue
            crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            pil_img  = Image.fromarray(crop_rgb)
            inp = preprocess(pil_img).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                feat = model.encode_image(inp)
                feat = F.normalize(feat, dim=-1)
            X_neg.append(feat.cpu().numpy()[0])

    X_pos = np.array(X_pos)
    X_neg = np.array(X_neg)

    n = min(len(X_pos), len(X_neg), max_pos)
    if n == 0:
        return np.array([]), np.array([])

    X = np.concatenate([X_pos[:n], X_neg[:n]], axis=0)
    y = np.concatenate([np.ones(n), np.zeros(n)])

    np.savez_compressed(cache_path, X=X, y=y)
    print(f'Сохранено {len(X)} эмбеддингов в {cache_path}')
    return X, y

## 11. Дистиллированный MLP-классификатор

In [ ]:
class SmokingMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=CLASSIFIER_HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_mlp_classifier(X_train, y_train, X_val, y_val,
                          epochs=CLASSIFIER_EPOCHS, lr=CLASSIFIER_LR, device=DEVICE):
    input_dim = X_train.shape[1]
    model = SmokingMLP(input_dim).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.BCEWithLogitsLoss()

    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_vl = torch.tensor(X_val,   dtype=torch.float32).to(device)
    y_vl = torch.tensor(y_val,   dtype=torch.float32).to(device)

    best_f1  = 0.0
    best_state = None

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(X_tr)
        loss = criterion(logits, y_tr)
        loss.backward()
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_vl)
            val_preds  = (torch.sigmoid(val_logits) >= 0.5).cpu().numpy().astype(int)
        val_f1 = f1_score(y_val.astype(int), val_preds, zero_division=0)
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())

        if (epoch + 1) % 5 == 0:
            print(f'  epoch {epoch+1}/{epochs}  loss={loss.item():.4f}  val_f1={val_f1:.4f}')

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f'Лучший val F1: {best_f1:.4f}')
    return model


def eval_mlp(model, X_test, y_test, threshold=0.5, device=DEVICE):
    model.eval()
    X_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = model(X_t)
        probs  = torch.sigmoid(logits).cpu().numpy()
    preds = (probs >= threshold).astype(int)
    p = precision_score(y_test.astype(int), preds, zero_division=0)
    r = recall_score(y_test.astype(int), preds, zero_division=0)
    f = f1_score(y_test.astype(int), preds, zero_division=0)
    return {'precision': p, 'recall': r, 'f1': f, 'probs': probs}

## 12. Утилиты оценки полного пайплайна (YOLO + VLM) на датасете

In [ ]:
def evaluate_pipeline_on_dataset(dataset_dir, split='test', smoking_cls_ids=None,
                                  vlm_name=None, prompt_set=None, vlm_threshold=0.5,
                                  mlp_model=None, vlm_entry_for_mlp=None,
                                  expand_factor=EXPAND_FACTOR,
                                  cig_conf=CIG_CONF_THRESH, final_conf=FINAL_CONF_THRESH,
                                  max_images=500):
    if smoking_cls_ids is None:
        smoking_cls_ids = {0}

    images_dir = Path(dataset_dir) / split / 'images'
    labels_dir = Path(dataset_dir) / split / 'labels'
    if not images_dir.exists():
        return None

    img_paths = sorted(images_dir.glob('*.jpg')) + sorted(images_dir.glob('*.png'))
    img_paths = img_paths[:max_images]

    tp = fp = fn = 0

    for img_path in tqdm(img_paths, desc=f'Eval {Path(dataset_dir).name}/{split}'):
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            continue
        h, w = img_bgr.shape[:2]

        label_path = labels_dir / (img_path.stem + '.txt')
        gt_boxes = [b['box'] for b in load_gt_boxes_yolo(label_path, w, h)
                    if b['cls'] in smoking_cls_ids]

        detections, _ = yolo_detect_image(img_path, cig_conf=cig_conf,
                                           final_conf=final_conf)

        if vlm_name is not None and vlm_name in loaded_vlms and len(detections) > 0:
            verified = vlm_verify_detections(
                img_bgr, detections, vlm_name, prompt_set,
                threshold=vlm_threshold, expand_factor=expand_factor
            )
            detections = [d for d, v in zip(detections, verified) if v]

        if mlp_model is not None and vlm_entry_for_mlp is not None and len(detections) > 0:
            feats = []
            valid_idx = []
            for i, det in enumerate(detections):
                crop = crop_and_upscale(img_bgr, det['box'], expand_factor)
                if crop is None:
                    continue
                crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                pil_img  = Image.fromarray(crop_rgb)
                inp = vlm_entry_for_mlp['preprocess'](pil_img).unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    feat = vlm_entry_for_mlp['model'].encode_image(inp)
                    feat = F.normalize(feat, dim=-1)
                feats.append(feat.cpu().numpy()[0])
                valid_idx.append(i)
            if feats:
                X_t = torch.tensor(np.array(feats), dtype=torch.float32).to(DEVICE)
                mlp_model.eval()
                with torch.no_grad():
                    probs = torch.sigmoid(mlp_model(X_t)).cpu().numpy()
                detections = [detections[i] for i, p in zip(valid_idx, probs) if p >= 0.5]

        pred_boxes = [d['box'] for d in detections]

        matched_gt = set()
        matched_pred = set()
        for pi, pb in enumerate(pred_boxes):
            best_iou = 0.0
            best_gi  = -1
            for gi, gb in enumerate(gt_boxes):
                if gi in matched_gt:
                    continue
                iou = compute_iou(pb, gb)
                if iou > best_iou:
                    best_iou = iou
                    best_gi  = gi
            if best_iou >= 0.5 and best_gi >= 0:
                tp += 1
                matched_gt.add(best_gi)
                matched_pred.add(pi)
            else:
                fp += 1

        fn += len(gt_boxes) - len(matched_gt)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return {'tp': tp, 'fp': fp, 'fn': fn,
            'precision': precision, 'recall': recall, 'f1': f1}

## 13. Подбор порогов: сетка по cig_conf и final_conf

In [ ]:
def threshold_grid_search(dataset_dir, split='val', smoking_cls_ids=None,
                           cig_confs=None, final_confs=None, max_images=300):
    if cig_confs is None:
        cig_confs = [0.10, 0.15, 0.20, 0.25, 0.30]
    if final_confs is None:
        final_confs = [0.15, 0.20, 0.25, 0.30, 0.35]

    results = []
    for cc in cig_confs:
        for fc in final_confs:
            if fc < cc:
                continue
            m = evaluate_pipeline_on_dataset(
                dataset_dir, split=split,
                smoking_cls_ids=smoking_cls_ids,
                vlm_name=None,
                cig_conf=cc, final_conf=fc,
                max_images=max_images
            )
            if m is None:
                continue
            results.append({
                'cig_conf': cc, 'final_conf': fc,
                **m
            })
            print(f'  cig={cc:.2f} final={fc:.2f} -> P={m["precision"]:.3f} R={m["recall"]:.3f} F1={m["f1"]:.3f}')

    df = pd.DataFrame(results)
    return df


print('Подбор порогов на micro датасете сигарет (val)...')
grid_df = threshold_grid_search(MICRO_DATASET_DIR, split='val', max_images=200)
grid_df.to_csv(OUTPUT_DIR / 'threshold_grid.csv', index=False)

# выбираем конфигурации с Precision >= 0.75, максимизируем Recall
candidates = grid_df[grid_df['precision'] >= 0.75].sort_values('recall', ascending=False)
print('\nКандидаты (Precision >= 0.75):')
print(candidates[['cig_conf', 'final_conf', 'precision', 'recall', 'f1']].head(5).to_string(index=False))

best_row = candidates.iloc[0] if len(candidates) > 0 else grid_df.sort_values('f1', ascending=False).iloc[0]
BEST_CIG_CONF   = float(best_row['cig_conf'])
BEST_FINAL_CONF = float(best_row['final_conf'])
print(f'\nВыбранные пороги: cig_conf={BEST_CIG_CONF}, final_conf={BEST_FINAL_CONF}')

## 14. Эксперимент 1: CLIP + порог VLM-верификации

In [ ]:
def vlm_threshold_sweep(dataset_dir, vlm_name, prompt_set, split='val',
                         thresholds=None, max_images=300):
    if thresholds is None:
        thresholds = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]
    results = []
    for thr in thresholds:
        m = evaluate_pipeline_on_dataset(
            dataset_dir, split=split,
            vlm_name=vlm_name, prompt_set=prompt_set,
            vlm_threshold=thr,
            cig_conf=BEST_CIG_CONF, final_conf=BEST_FINAL_CONF,
            max_images=max_images
        )
        if m is None:
            continue
        results.append({'vlm': vlm_name, 'prompt_set': prompt_set, 'threshold': thr, **m})
        print(f'  {vlm_name}/{prompt_set} thr={thr:.2f} -> P={m["precision"]:.3f} R={m["recall"]:.3f} F1={m["f1"]:.3f}')
    return pd.DataFrame(results)


vlm_sweep_frames = []
for vlm_name in loaded_vlms:
    for pset in ['basic', 'detailed']:
        print(f'\nVLM sweep: {vlm_name} / {pset}')
        df_sw = vlm_threshold_sweep(MICRO_DATASET_DIR, vlm_name, pset, max_images=200)
        vlm_sweep_frames.append(df_sw)

if vlm_sweep_frames:
    vlm_sweep_df = pd.concat(vlm_sweep_frames, ignore_index=True)
    vlm_sweep_df.to_csv(OUTPUT_DIR / 'vlm_threshold_sweep.csv', index=False)

    best_vlm_row = (
        vlm_sweep_df[vlm_sweep_df['precision'] >= 0.75]
        .sort_values('recall', ascending=False)
    )
    if len(best_vlm_row) > 0:
        best_vlm_row = best_vlm_row.iloc[0]
        BEST_VLM_NAME       = best_vlm_row['vlm']
        BEST_PROMPT_SET     = best_vlm_row['prompt_set']
        BEST_VLM_THRESHOLD  = float(best_vlm_row['threshold'])
        print(f'\nЛучший VLM: {BEST_VLM_NAME} / {BEST_PROMPT_SET} / thr={BEST_VLM_THRESHOLD}')
    else:
        # берём лучший по F1
        best_vlm_row = vlm_sweep_df.sort_values('f1', ascending=False).iloc[0]
        BEST_VLM_NAME      = best_vlm_row['vlm']
        BEST_PROMPT_SET    = best_vlm_row['prompt_set']
        BEST_VLM_THRESHOLD = float(best_vlm_row['threshold'])
        print(f'Нет конфигурации с P>=0.75, лучший по F1: {BEST_VLM_NAME}')

## 15. Эксперимент 2: дистилляция MLP на CLIP-эмбеддингах

In [ ]:
trained_mlps = {}

for vlm_name in loaded_vlms:
    print(f'\n=== Дистилляция для {vlm_name} ===')

    X_train, y_train = collect_embed_dataset(
        SMALL_DATASET_DIR, vlm_name, prompt_set_name='basic', split='train',
        max_pos=1500, max_neg=1500
    )
    X_val, y_val = collect_embed_dataset(
        SMALL_DATASET_DIR, vlm_name, prompt_set_name='basic', split='val',
        max_pos=500, max_neg=500
    )
    X_test, y_test = collect_embed_dataset(
        SMALL_DATASET_DIR, vlm_name, prompt_set_name='basic', split='test',
        max_pos=500, max_neg=500
    )

    if len(X_train) == 0 or len(X_val) == 0:
        print('Недостаточно данных, пропускаю')
        continue

    # нормализация
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s   = scaler.transform(X_val)
    X_test_s  = scaler.transform(X_test) if len(X_test) > 0 else X_val_s

    # --- LogReg baseline ---
    lr_clf = LogisticRegression(max_iter=500, C=1.0)
    lr_clf.fit(X_train_s, y_train.astype(int))
    lr_preds = lr_clf.predict(X_test_s)
    lr_p = precision_score(y_test.astype(int) if len(X_test) > 0 else y_val.astype(int),
                            lr_preds, zero_division=0)
    lr_r = recall_score(y_test.astype(int) if len(X_test) > 0 else y_val.astype(int),
                         lr_preds, zero_division=0)
    lr_f = f1_score(y_test.astype(int) if len(X_test) > 0 else y_val.astype(int),
                     lr_preds, zero_division=0)
    print(f'LogReg baseline -> P={lr_p:.3f} R={lr_r:.3f} F1={lr_f:.3f}')

    # --- MLP ---
    mlp = train_mlp_classifier(X_train_s, y_train, X_val_s, y_val)
    y_test_eval = y_test if len(X_test) > 0 else y_val
    mlp_metrics = eval_mlp(mlp, X_test_s, y_test_eval)
    print(f'MLP -> P={mlp_metrics["precision"]:.3f} R={mlp_metrics["recall"]:.3f} F1={mlp_metrics["f1"]:.3f}')

    torch.save(mlp.state_dict(), OUTPUT_DIR / f'mlp_{vlm_name}.pt')
    trained_mlps[vlm_name] = {
        'model': mlp,
        'scaler': scaler,
        'vlm_entry': loaded_vlms[vlm_name],
        'metrics': mlp_metrics
    }

## 16. Финальная оценка всех конфигураций на test-сплите

In [ ]:
EVAL_DATASETS = [
    {'name': 'cig_micro', 'path': MICRO_DATASET_DIR, 'split': 'test'},
    {'name': 'cig_small', 'path': SMALL_DATASET_DIR, 'split': 'test'},
    {'name': 'smoke_micro', 'path': SMOKE_MICRO_DIR, 'split': 'test'},
    {'name': 'smoke_small', 'path': SMOKE_SMALL_DIR, 'split': 'test'},
]

all_results = []


def run_eval_config(config_name, dataset_info, **kwargs):
    m = evaluate_pipeline_on_dataset(
        dataset_info['path'],
        split=dataset_info['split'],
        max_images=400,
        **kwargs
    )
    if m is None:
        return
    all_results.append({
        'config': config_name,
        'dataset': dataset_info['name'],
        **m
    })
    print(f'  [{config_name}] {dataset_info["name"]} -> P={m["precision"]:.3f} R={m["recall"]:.3f} F1={m["f1"]:.3f}')


for ds in EVAL_DATASETS:
    print(f'\n--- Базовый пайплайн (без VLM) ---')
    run_eval_config('baseline', ds,
                    cig_conf=CIG_CONF_THRESH, final_conf=FINAL_CONF_THRESH)

    print(f'--- Оптимальные пороги (без VLM) ---')
    run_eval_config('tuned_yolo', ds,
                    cig_conf=BEST_CIG_CONF, final_conf=BEST_FINAL_CONF)

    for vlm_name in loaded_vlms:
        for pset in ['basic', 'detailed']:
            print(f'--- VLM: {vlm_name}/{pset} ---')
            run_eval_config(f'vlm_{vlm_name}_{pset}', ds,
                            cig_conf=BEST_CIG_CONF, final_conf=BEST_FINAL_CONF,
                            vlm_name=vlm_name, prompt_set=pset,
                            vlm_threshold=BEST_VLM_THRESHOLD
                              if vlm_name == BEST_VLM_NAME else 0.5)

    for vlm_name, mlp_info in trained_mlps.items():
        print(f'--- MLP ({vlm_name}) ---')
        run_eval_config(f'mlp_{vlm_name}', ds,
                        cig_conf=BEST_CIG_CONF, final_conf=BEST_FINAL_CONF,
                        mlp_model=mlp_info['model'],
                        vlm_entry_for_mlp=mlp_info['vlm_entry'])


results_df = pd.DataFrame(all_results)
results_df.to_csv(OUTPUT_DIR / 'final_results.csv', index=False)
print('\nСохранено в', OUTPUT_DIR / 'final_results.csv')

## 17. Сводная таблица и визуализация

In [ ]:
pivot_p = results_df.pivot(index='config', columns='dataset', values='precision').round(3)
pivot_r = results_df.pivot(index='config', columns='dataset', values='recall').round(3)
pivot_f = results_df.pivot(index='config', columns='dataset', values='f1').round(3)

print('=== Precision ===')
print(pivot_p.to_string())
print('\n=== Recall ===')
print(pivot_r.to_string())
print('\n=== F1 ===')
print(pivot_f.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle('Сравнение конфигураций VLM-валидации', fontsize=14, fontweight='bold')

for ax, (pivot, title) in zip(axes, [
    (pivot_p, 'Precision'),
    (pivot_r, 'Recall'),
    (pivot_f, 'F1'),
]):
    pivot.plot(kind='bar', ax=ax, width=0.75)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.axhline(0.75, color='red', linestyle='--', alpha=0.6, label='Precision 0.75')
    ax.grid(axis='y', alpha=0.3)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=7)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'vlm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 18. PR-кривая для MLP-классификатора

In [ ]:
from sklearn.metrics import precision_recall_curve

fig, axes = plt.subplots(1, len(trained_mlps), figsize=(6 * len(trained_mlps), 5))
if len(trained_mlps) == 1:
    axes = [axes]

for ax, (vlm_name, mlp_info) in zip(axes, trained_mlps.items()):
    probs  = mlp_info['metrics']['probs']
    y_test_key = 'test' if (SMALL_DATASET_DIR / 'test' / 'images').exists() else 'val'
    _, y_gt = collect_embed_dataset(SMALL_DATASET_DIR, vlm_name, split=y_test_key,
                                     max_pos=500, max_neg=500)
    if len(probs) != len(y_gt):
        continue

    prec, rec, thr = precision_recall_curve(y_gt.astype(int), probs)
    ax.plot(rec, prec, lw=2)
    ax.axhline(0.75, color='red', linestyle='--', alpha=0.6)
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(f'PR-кривая MLP ({vlm_name})')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pr_curves_mlp.png', dpi=150, bbox_inches='tight')
plt.show()

## 19. Визуальный инференс — примеры работы лучшей конфигурации

In [ ]:
def draw_detections(img_bgr, detections, color=(0, 255, 0), label_prefix=''):
    out = img_bgr.copy()
    for det in detections:
        x1, y1, x2, y2 = map(int, det['box'])
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        txt = f'{label_prefix}{det["conf"]:.2f}'
        cv2.putText(out, txt, (x1, max(0, y1 - 4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    return out


sample_images_dir = MICRO_DATASET_DIR / 'test' / 'images'
if not sample_images_dir.exists():
    sample_images_dir = SMALL_DATASET_DIR / 'test' / 'images'

sample_paths = random.sample(
    list(sample_images_dir.glob('*.jpg')) + list(sample_images_dir.glob('*.png')),
    k=min(6, len(list(sample_images_dir.glob('*.*'))))
)

best_mlp_name = max(trained_mlps, key=lambda k: trained_mlps[k]['metrics']['f1']) \
    if trained_mlps else None

fig, axes = plt.subplots(len(sample_paths), 2, figsize=(14, 5 * len(sample_paths)))
if len(sample_paths) == 1:
    axes = [axes]

for ax_row, img_path in zip(axes, sample_paths):
    dets_base, img_bgr = yolo_detect_image(
        img_path, cig_conf=BEST_CIG_CONF, final_conf=BEST_FINAL_CONF
    )
    if img_bgr is None:
        continue

    vis_base = draw_detections(img_bgr, dets_base, color=(0, 200, 0), label_prefix='YOLO ')
    ax_row[0].imshow(cv2.cvtColor(vis_base, cv2.COLOR_BGR2RGB))
    ax_row[0].set_title(f'YOLO: {len(dets_base)} det')
    ax_row[0].axis('off')

    dets_vlm = list(dets_base)
    if best_mlp_name is not None:
        mlp_info = trained_mlps[best_mlp_name]
        if len(dets_vlm) > 0:
            feats = []
            valid_idx = []
            for i, det in enumerate(dets_vlm):
                crop = crop_and_upscale(img_bgr, det['box'])
                if crop is None:
                    continue
                crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                pil_img  = Image.fromarray(crop_rgb)
                inp = mlp_info['vlm_entry']['preprocess'](pil_img).unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    feat = mlp_info['vlm_entry']['model'].encode_image(inp)
                    feat = F.normalize(feat, dim=-1)
                feats.append(feat.cpu().numpy()[0])
                valid_idx.append(i)
            if feats:
                X_t = torch.tensor(
                    mlp_info['scaler'].transform(np.array(feats)),
                    dtype=torch.float32
                ).to(DEVICE)
                mlp_info['model'].eval()
                with torch.no_grad():
                    probs = torch.sigmoid(mlp_info['model'](X_t)).cpu().numpy()
                dets_vlm = [dets_vlm[i] for i, p in zip(valid_idx, probs) if p >= 0.5]

    vis_vlm = draw_detections(img_bgr, dets_vlm, color=(0, 100, 255), label_prefix='VLM ')
    ax_row[1].imshow(cv2.cvtColor(vis_vlm, cv2.COLOR_BGR2RGB))
    ax_row[1].set_title(f'YOLO+MLP: {len(dets_vlm)} det')
    ax_row[1].axis('off')

plt.suptitle('Сравнение: базовый YOLO vs YOLO+VLM(MLP)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'visual_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 20. Итоговый вывод

In [ ]:
print('='*60)
print('ИТОГОВЫЕ РЕЗУЛЬТАТЫ')
print('='*60)

summary = (
    results_df
    .groupby('config')[['precision', 'recall', 'f1']]
    .mean()
    .round(3)
    .sort_values('f1', ascending=False)
)
print(summary.to_string())

valid_configs = summary[summary['precision'] >= 0.75]
if len(valid_configs) > 0:
    best_config = valid_configs.sort_values('recall', ascending=False).index[0]
    row = valid_configs.loc[best_config]
    print(f'\nЛучшая конфигурация (P>=0.75, max Recall):')
    print(f'  {best_config}')
    print(f'  Precision = {row["precision"]:.3f}')
    print(f'  Recall    = {row["recall"]:.3f}')
    print(f'  F1        = {row["f1"]:.3f}')
else:
    best_config = summary.index[0]
    row = summary.iloc[0]
    print(f'\nЛучшая конфигурация по F1 (нет вариантов с P>=0.75):')
    print(f'  {best_config}')
    print(f'  Precision = {row["precision"]:.3f}')
    print(f'  Recall    = {row["recall"]:.3f}')
    print(f'  F1        = {row["f1"]:.3f}')

summary.to_csv(OUTPUT_DIR / 'summary.csv')
print(f'\nВсе результаты сохранены в {OUTPUT_DIR}/')